In [45]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

In [46]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GOOGLE_GENAI_KEY"),
)

In [47]:
#Create a Satte
class LLMState(TypedDict):
    question: str
    answer: str

In [76]:
def llm_qa(state: LLMState) -> LLMState:
    #Extract question from state
    question = state['question']
    #Form the prompt
    prompt = f'Answer the following question: {question}'
    #Ask the prompt to the LLM and get the answer
    answer = llm.invoke(prompt).content[0]['text']
    #Update the answer
    state['answer'] = answer
    return state

In [77]:
#Create a StateGraph
graph = StateGraph(LLMState)

#ADD nodes
graph.add_node('llm_qa', llm_qa)

#ADD Edge
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

#Compile the graph into a workflow
workflow = graph.compile()

In [78]:
initial_state = {
    'question': 'What is the capital of Nepal?',
    'answer': ''
}

final_state = workflow.invoke(initial_state)

print("Final State:", final_state)


Final State: {'question': 'What is the capital of Nepal?', 'answer': 'The capital of Nepal is Kathmandu.'}
